In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# ══════════════════════════════════════════════════════════════════
# LINEAR REGRESSION — UNIVERSAL INTERPRETATION TOOLKIT
# ══════════════════════════════════════════════════════════════════
# HOW TO USE:
#   1. Train your model normally
#   2. Fill in the CONFIG block below
#   3. Run — everything is printed and plotted automatically
# ══════════════════════════════════════════════════════════════════


# ──────────────────────────────────────────────────────────────────
# CONFIG — change only this block
# ──────────────────────────────────────────────────────────────────

# Your trained model (must be sklearn LinearRegression)
# model = your_trained_model

# Your feature dataframe (X) and target series (y)
# X = your_X_dataframe
# y = your_y_series

# Feature names as a list — must match column order in X
# feature_names = ['age', 'bmi', 'smoker', ...]

# Target name (for labels)
# target_name = 'charges'

# Currency / unit symbol shown on plots
# unit = '$'

# If you used StandardScaler — pass the fitted scaler
# Otherwise set to None
# scaler = your_fitted_scaler   or   scaler = None

# One person to explain (dict matching feature names)
# person_to_explain = {'age': 45, 'bmi': 32, 'smoker': 1, ...}

# ──────────────────────────────────────────────────────────────────
# DEMO — remove this block when using your own model
# ──────────────────────────────────────────────────────────────────
import seaborn as sns

df            = sns.load_dataset('healthexp')          # placeholder
df            = pd.read_csv('insurance.csv')           # your real data
df['smoker']  = (df['smoker'] == 'yes').astype(int)
df            = pd.get_dummies(df, columns=['region','sex'],
                               drop_first=True)

feature_names = ['age','bmi','children','smoker']
target_name   = 'charges'
unit          = '$'
scaler        = None

X = df[feature_names].astype(float)
y = df[target_name].astype(float)

model = LinearRegression().fit(X, y)

person_to_explain = {
    'age':      45,
    'bmi':      32.0,
    'children': 2,
    'smoker':   1
}
# ──────────────────────────────────────────────────────────────────


# ══════════════════════════════════════════════════════════════════
# SECTION 1 — Extract and unscale coefficients
# ══════════════════════════════════════════════════════════════════

def get_real_coefficients(model, feature_names, scaler=None):
    """
    If scaler was used — convert coefficients back to original units.
    If no scaler     — return coefficients as-is.
    """
    coefs = model.coef_.copy()

    if scaler is not None:
        # coef in SD units ÷ SD of feature = coef in original units
        coefs = coefs / scaler.scale_

    return pd.DataFrame({
        'Feature'    : feature_names,
        'Coefficient': coefs,
        'Direction'  : ['Increases ' + target_name if c > 0
                        else 'Decreases ' + target_name
                        for c in coefs],
        'Abs_Value'  : np.abs(coefs)
    }).sort_values('Abs_Value', ascending=False).reset_index(drop=True)


coef_df = get_real_coefficients(model, feature_names, scaler)


# ══════════════════════════════════════════════════════════════════
# SECTION 2 — Print coefficient table
# ══════════════════════════════════════════════════════════════════

def print_coefficients(coef_df, model, unit, target_name):
    print("\n" + "═" * 62)
    print("  COEFFICIENT INTERPRETATION TABLE")
    print("═" * 62)
    print(f"  Base value (β₀ intercept) : "
          f"{unit}{model.intercept_:>12,.2f}")
    print(f"  → Starting {target_name} before any feature is applied")
    print("─" * 62)
    print(f"  {'Feature':<20} {'Coefficient':>14}  {'Meaning'}")
    print("─" * 62)

    for _, row in coef_df.iterrows():
        coef  = row['Coefficient']
        sign  = '+' if coef >= 0 else '-'
        label = ('adds' if coef >= 0 else 'saves')
        print(f"  {row['Feature']:<20} "
              f"{sign}{unit}{abs(coef):>12,.2f}  "
              f"← each unit {label} "
              f"{unit}{abs(coef):,.2f}")

    print("─" * 62)
    print(f"\n  Ranked by importance (largest absolute effect first):\n")
    for rank, (_, row) in enumerate(coef_df.iterrows(), 1):
        bar = '█' * int(row['Abs_Value'] /
                        coef_df['Abs_Value'].max() * 20)
        print(f"  {rank}. {row['Feature']:<18} {bar}")
    print("═" * 62)


print_coefficients(coef_df, model, unit, target_name)


# ══════════════════════════════════════════════════════════════════
# SECTION 3 — Explain one specific person
# ══════════════════════════════════════════════════════════════════

def explain_person(person_dict, model, feature_names,
                   coef_df, unit, target_name, scaler=None):

    print("\n" + "═" * 62)
    print("  SINGLE PREDICTION BREAKDOWN")
    print("═" * 62)
    print("\n  Profile:")
    for k, v in person_dict.items():
        print(f"    {k:<18} : {v}")

    # Build input
    person_df = pd.DataFrame([person_dict])[feature_names].astype(float)

    # Scale if needed
    person_input = scaler.transform(person_df) if scaler else person_df.values
    prediction   = model.predict(person_input)[0]

    # Contributions — always in original units
    real_coefs = coef_df.set_index('Feature')['Coefficient']

    print(f"\n  Charge breakdown:")
    print(f"  {'─'*56}")
    print(f"  {'Component':<28} {'Calculation':>14} {'Amount':>12}")
    print(f"  {'─'*56}")
    print(f"  {'Base price (β₀)':<28} {'':>14} "
          f"{unit}{model.intercept_:>10,.2f}")

    total_check = model.intercept_
    contributions = {}

    for feat in feature_names:
        val        = person_dict[feat]
        coef       = real_coefs[feat]
        contrib    = coef * val
        total_check += contrib
        contributions[feat] = contrib
        calc_str   = f"{val} × {unit}{coef:,.0f}"
        print(f"  {feat:<28} {calc_str:>14} "
              f"{unit}{contrib:>10,.2f}")

    print(f"  {'─'*56}")
    print(f"  {'PREDICTED ' + target_name.upper():<28} "
          f"{'':>14} {unit}{prediction:>10,.2f}")
    print(f"  {'═'*56}")

    # Biggest driver
    biggest = max(contributions, key=lambda k: abs(contributions[k]))
    print(f"\n  Biggest driver  : {biggest} "
          f"({unit}{contributions[biggest]:,.2f})")
    print(f"  Smallest driver : "
          f"{min(contributions, key=lambda k: abs(contributions[k]))}")

    return contributions, prediction


contributions, prediction = explain_person(
    person_to_explain, model, feature_names,
    coef_df, unit, target_name, scaler
)


# ══════════════════════════════════════════════════════════════════
# SECTION 4 — Compare two people
# ══════════════════════════════════════════════════════════════════

def compare_two_people(person_a, person_b,
                       model, feature_names,
                       coef_df, unit, target_name, scaler=None):
    """
    Pass two person dicts — shows exactly which features
    drive the difference in predicted price.
    """
    real_coefs = coef_df.set_index('Feature')['Coefficient']

    def predict(p):
        df_  = pd.DataFrame([p])[feature_names].astype(float)
        inp  = scaler.transform(df_) if scaler else df_.values
        return model.predict(inp)[0]

    pred_a = predict(person_a)
    pred_b = predict(person_b)

    print("\n" + "═" * 62)
    print("  COMPARISON — TWO PEOPLE")
    print("═" * 62)
    print(f"\n  {'Feature':<20} {'Person A':>12} {'Person B':>12} "
          f"{'Diff effect':>14}")
    print(f"  {'─'*60}")

    for feat in feature_names:
        va      = person_a[feat]
        vb      = person_b[feat]
        coef    = real_coefs[feat]
        effect  = (vb - va) * coef
        marker  = ' ←' if abs(effect) > 1000 else ''
        print(f"  {feat:<20} {va:>12} {vb:>12} "
              f"  {unit}{effect:>10,.2f}{marker}")

    print(f"  {'─'*60}")
    print(f"  {'Predicted ' + target_name:<20} "
          f"{unit}{pred_a:>10,.2f}   {unit}{pred_b:>10,.2f}")
    diff = pred_b - pred_a
    print(f"\n  Net difference  : {unit}{diff:,.2f}  "
          f"({'Person B pays more' if diff > 0 else 'Person A pays more'})")
    print("═" * 62)


# Example — same person, smoker vs non-smoker
person_a = {**person_to_explain, 'smoker': 0}
person_b = {**person_to_explain, 'smoker': 1}
compare_two_people(person_a, person_b, model, feature_names,
                   coef_df, unit, target_name, scaler)


# ══════════════════════════════════════════════════════════════════
# SECTION 5 — What-if analysis: change one feature, see the effect
# ══════════════════════════════════════════════════════════════════

def whatif_analysis(person_dict, feature_to_vary, values_to_try,
                    model, feature_names, scaler=None):
    """
    Holds everything constant, varies one feature.
    Shows exactly how that one feature moves the prediction.
    """
    print("\n" + "═" * 62)
    print(f"  WHAT-IF — varying '{feature_to_vary}'")
    print("═" * 62)
    print(f"  (all other features held constant)\n")
    print(f"  {feature_to_vary:<20} {'Predicted ' + target_name:>18}  Change from base")
    print(f"  {'─'*56}")

    base        = {**person_dict}
    base_df     = pd.DataFrame([base])[feature_names].astype(float)
    base_input  = scaler.transform(base_df) if scaler else base_df.values
    base_pred   = model.predict(base_input)[0]

    for val in values_to_try:
        variant     = {**person_dict, feature_to_vary: val}
        var_df      = pd.DataFrame([variant])[feature_names].astype(float)
        var_input   = scaler.transform(var_df) if scaler else var_df.values
        pred        = model.predict(var_input)[0]
        delta       = pred - base_pred
        marker      = ' ▲' if delta > 0 else ' ▼'
        print(f"  {str(val):<20} {unit}{pred:>16,.2f}  "
              f"{'+' if delta>=0 else ''}{unit}{delta:,.2f}{marker}")

    print("═" * 62)


# Example: what happens as age increases?
whatif_analysis(
    person_to_explain,
    feature_to_vary  = 'age',
    values_to_try    = [25, 30, 35, 40, 45, 50, 55, 60],
    model            = model,
    feature_names    = feature_names,
    scaler           = scaler
)

# Example: what happens as bmi increases?
whatif_analysis(
    person_to_explain,
    feature_to_vary  = 'bmi',
    values_to_try    = [18.5, 22, 25, 28, 30, 35, 40],
    model            = model,
    feature_names    = feature_names,
    scaler           = scaler
)


# ══════════════════════════════════════════════════════════════════
# SECTION 6 — Plots (4 visuals in one figure)
# ══════════════════════════════════════════════════════════════════

def plot_interpretation(coef_df, contributions, prediction,
                        model, feature_names, unit, target_name,
                        person_to_explain):

    fig = plt.figure(figsize=(15, 11))
    fig.suptitle('Linear Regression — Full Interpretation Dashboard',
                 fontsize=14, y=1.01)
    gs  = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

    # ── Plot 1: Coefficient bar chart ─────────────────────────────
    ax1   = fig.add_subplot(gs[0, 0])
    coefs = coef_df['Coefficient'].values
    feats = coef_df['Feature'].values
    cols  = ['#2196a6' if c > 0 else '#d64e3a' for c in coefs]

    bars  = ax1.barh(feats, coefs, color=cols, alpha=0.85,
                     edgecolor='white', height=0.6)
    ax1.axvline(0, color='black', linewidth=0.8)
    ax1.set_xlabel(f'Effect per unit ({unit})')
    ax1.set_title('Coefficient values\n(blue = increases, red = decreases)')

    for bar, val in zip(bars, coefs):
        xpos  = val + max(abs(coefs)) * 0.02 if val >= 0 \
                else val - max(abs(coefs)) * 0.02
        align = 'left' if val >= 0 else 'right'
        ax1.text(xpos, bar.get_y() + bar.get_height() / 2,
                 f'{unit}{val:,.0f}', va='center',
                 ha=align, fontsize=9)

    # ── Plot 2: Waterfall — one person's breakdown ────────────────
    ax2       = fig.add_subplot(gs[0, 1])
    labels    = ['Base'] + list(contributions.keys()) + ['Predicted']
    base      = model.intercept_
    contribs  = list(contributions.values())
    final     = prediction

    running   = base
    bottoms   = [0]
    heights   = [base]
    bar_cols  = ['#5a9bd4']

    for c in contribs:
        bottoms.append(running if c >= 0 else running + c)
        heights.append(abs(c))
        bar_cols.append('#2196a6' if c >= 0 else '#d64e3a')
        running += c

    bottoms.append(0)
    heights.append(final)
    bar_cols.append('#27ae60')

    ax2.bar(labels, heights, bottom=bottoms,
            color=bar_cols, alpha=0.85, edgecolor='white', width=0.6)
    ax2.set_ylabel(f'Predicted {target_name} ({unit})')
    ax2.set_title('Waterfall — prediction breakdown\nfor the profiled person')
    ax2.tick_params(axis='x', rotation=30)

    for i, (b, h) in enumerate(zip(bottoms, heights)):
        ax2.text(i, b + h + final * 0.01, f'{unit}{b+h:,.0f}',
                 ha='center', va='bottom', fontsize=8)

    # ── Plot 3: Feature importance (absolute effect) ──────────────
    ax3      = fig.add_subplot(gs[1, 0])
    imp_df   = coef_df.sort_values('Abs_Value')
    norm_imp = imp_df['Abs_Value'] / imp_df['Abs_Value'].sum() * 100

    ax3.barh(imp_df['Feature'], norm_imp,
             color='#7b68c8', alpha=0.8, edgecolor='white', height=0.6)
    ax3.set_xlabel('Relative importance (%)')
    ax3.set_title('Feature importance\n(% of total coefficient magnitude)')

    for i, (feat, val) in enumerate(
            zip(imp_df['Feature'], norm_imp)):
        ax3.text(val + 0.5, i, f'{val:.1f}%',
                 va='center', fontsize=9)

    # ── Plot 4: What-if — age effect ──────────────────────────────
    ax4       = fig.add_subplot(gs[1, 1])
    vary_feat = feature_names[0]         # first feature — change if needed
    feat_min  = float(X[vary_feat].min())
    feat_max  = float(X[vary_feat].max())
    feat_vals = np.linspace(feat_min, feat_max, 60)
    preds     = []

    for val in feat_vals:
        p      = {**person_to_explain, vary_feat: val}
        p_df   = pd.DataFrame([p])[feature_names].astype(float)
        p_inp  = scaler.transform(p_df) if scaler else p_df.values
        preds.append(model.predict(p_inp)[0])

    ax4.plot(feat_vals, preds, color='#2196a6', linewidth=2.2)
    ax4.axvline(person_to_explain[vary_feat],
                color='tomato', linestyle='--',
                linewidth=1.5, label='Person value')
    ax4.set_xlabel(vary_feat)
    ax4.set_ylabel(f'Predicted {target_name} ({unit})')
    ax4.set_title(f'What-if — varying {vary_feat}\n'
                  f'(all else held constant)')
    ax4.legend(fontsize=9)

    plt.tight_layout()
    plt.show()


plot_interpretation(
    coef_df, contributions, prediction,
    model, feature_names, unit, target_name,
    person_to_explain
)

print("\n✓ Interpretation complete.")
```

---

## What each section gives you
```
SECTION 1 — get_real_coefficients()
  Extracts all β values, unscales if needed,
  ranks by importance automatically.

SECTION 2 — print_coefficients()
  Prints a full table with plain-English meaning
  and a text bar chart of importance.

SECTION 3 — explain_person()
  Takes one person → shows exactly how each
  feature contributed to their final prediction.

SECTION 4 — compare_two_people()
  Side-by-side comparison → shows which features
  drive the price difference between two profiles.

SECTION 5 — whatif_analysis()
  Varies one feature across a range, holds
  everything else constant → shows sensitivity.

SECTION 6 — plot_interpretation()
  4 plots: coefficients, waterfall breakdown,
  feature importance %, and what-if curve.